# The Halting Diagonal — Predictions

**Pre-registration date:** 2026-07-29
**Rule:** All predictions entered here BEFORE the engine is run. Results —
confirmed or failed — remain in the record. Failed predictions stay in the
data, period, full stop.

Every prediction below is **decidable by running code in finite time**. There
are no conjectures in this paper and no open problems. A prediction that cannot
be settled by execution does not belong here.

---

## P1 — The diagonal escapes, and the check is cheap

**Claim.** For every finite halting table `T` of order `n`, the constructed row
`D[i] = 1 − T[i][i]` differs from row `i` at column `i` for all `i`, hence
`D ∉ rows(T)`.

Further: the property depends only on `diag(T)`, so exhaustive verification
over all tables of order `n` reduces from `2^(n²)` to `2ⁿ`.

**Falsification.** Exhibit any `n` and any diagonal in `{0,1}ⁿ` for which the
escape fails, or any table `T` whose rows contain `D`. One counterexample kills
P1.

**Test.** Exhaustive over all `2ⁿ` diagonals for `n = 1..16`, plus 2000 random
full tables of order 2..11.

## P2 — The derangement count is exact, by two independent algorithms

**Claim.** `D_n`, the number of permutations of `n` elements with no fixed
point, satisfies

```
D_n = n! · Σ_{k=0..n} (−1)^k / k!        (inclusion-exclusion)
D_n = (n−1)·(D_{n−1} + D_{n−2})           (recurrence)
D_n = round(n!/e)                          (closed form, n ≥ 1)
```

and all three agree **exactly**, in integer arithmetic, with no floating-point
rounding in the first two.

**Falsification.** Any `n` at which the three disagree.

**Test.** `n = 0..60` for the first two; `n = 1..17` for the closed form
(bounded by float64 range, not by the claim).

## P3 — The involution is order 4 with no fixed point, at every level

**Claim.** In the 2×2 real representation of ℂ: `i² = −I₂` and `i⁴ = +I₂`
exactly, `det(−I₂) = +1`, `trace(−I₂) = −2`, and `−I₂` has eigenvalues `−1, −1`
— no fixed vector but the origin.

In the Cayley–Dickson tower at the sedenion level: `eₖ² = −1` for exactly
`k = 1..15`, and `e₀` is the **unique** basis element with `e₀² = +1`.

**Falsification.** Any basis element outside `k = 1..15` squaring to `−1`, or
any second element squaring to `+1`, or the count differing from 15.

**Test.** Direct matrix powers; direct Cayley–Dickson multiplication of all 16
basis elements.

## P4 — The no-fixed-point constraint prunes the crib search measurably

**Claim.** Because the Enigma reflector guarantees `cipher[k] ≠ plain[k]`, any
crib alignment in which the ciphertext agrees with the crib at even one
position is impossible and is discarded with no rotor work at all. For a crib
of length `L` over an alphabet of size `A`, the surviving fraction of
alignments is

```
(1 − 1/A)^L
```

**Falsification.** Measured survival departing from `(1 − 1/A)^L` by more than
2% at any tested `L`.

**Test.** `A = 26`, ciphertext of 200,000 symbols, `L ∈ {4, 8, 12, 16, 20, 25}`,
by direct enumeration of every alignment.

---

## Registered as executable assertions

Below, each prediction is a callable returning `True`/`False`. They are
**defined** here and **run** in `03_results.ipynb`, not before.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.turing_diagonal import maths as td
from ValaQuenta import zero_lattice as zl

import math, itertools
from fractions import Fraction
import numpy as np

print('engine   : ValaQuenta.modules.turing_diagonal')
print('python   :', sys.version.split()[0])

In [ ]:
# Predictions registered as assertions. Defined now, evaluated in 03.

def P1_diagonal_escape(n_max=16, n_random=2000):
    for n in range(1, n_max + 1):
        for diag in itertools.product((0, 1), repeat=n):
            D = [1 - d for d in diag]
            if any(D[i] == diag[i] for i in range(n)):
                return False
    rng = np.random.default_rng(0)
    for _ in range(n_random):
        n = int(rng.integers(2, 12))
        T = rng.integers(0, 2, size=(n, n))
        D = 1 - np.diag(T)
        if any(np.array_equal(D, T[i]) for i in range(n)):
            return False
    return True


def _D_series(m):
    return int(sum(Fraction((-1) ** k, math.factorial(k))
                   for k in range(m + 1)) * math.factorial(m))


def _D_recur(m):
    a, b = 1, 0                      # D_0 = 1, D_1 = 0
    if m == 0: return a
    if m == 1: return b
    for k in range(2, m + 1):
        a, b = b, (k - 1) * (b + a)
    return b


def P2_derangement_exact(m_max=60):
    if any(_D_series(m) != _D_recur(m) for m in range(m_max + 1)):
        return False
    return all(_D_recur(m) == round(math.factorial(m) / math.e)
               for m in range(1, 18))


def P3_involution():
    I2 = np.eye(2)
    i_m = np.array([[0.0, -1.0], [1.0, 0.0]])
    if not np.array_equal(np.linalg.matrix_power(i_m, 2), -I2): return False
    if not np.array_equal(np.linalg.matrix_power(i_m, 4),  I2): return False
    sq = [zl.multiply(zl.e_k(k), zl.e_k(k))[0] for k in range(16)]
    neg = [k for k in range(16) if abs(sq[k] + 1) < 1e-12]
    pos = [k for k in range(16) if abs(sq[k] - 1) < 1e-12]
    return neg == list(range(1, 16)) and pos == [0]


def P4_crib_pruning(A=26, N=200_000, tol=0.02):
    rng = np.random.default_rng(20260606)
    cipher = rng.integers(0, A, size=N)
    for L in (4, 8, 12, 16, 20, 25):
        crib = rng.integers(0, A, size=L)
        n_align = N - L + 1
        surv = sum(1 for s in range(n_align)
                   if not np.any(cipher[s:s + L] == crib))
        obs, pred = surv / n_align, (1 - 1 / A) ** L
        if abs(obs - pred) / pred > tol:
            return False
    return True


PREDICTIONS = [
    ('P1', 'Diagonal escapes every table; check is O(n) on diag only', P1_diagonal_escape),
    ('P2', 'D_n exact by three independent routes',                     P2_derangement_exact),
    ('P3', 'Involution order 4, no fixed point; e_0 unique at +1',      P3_involution),
    ('P4', 'Derangement prunes crib alignments to (1-1/A)^L',           P4_crib_pruning),
]

print(f'{len(PREDICTIONS)} predictions registered, none evaluated yet:')
for tag, desc, _ in PREDICTIONS:
    print(f'  {tag}  {desc}')